# Description

In this notebook, I will:
- Load the pre-trained model.
- Fine-tuning it on if-dataset

In [1]:
import os, json, torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from transformers import (
    PreTrainedTokenizerFast,
    BertForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

In [ ]:
DATA_DIR = "if_dataset"               # where train.jsonl / validation.jsonl / test.jsonl live
MODEL_DIR = "mlm_model_bert/checkpoint-50000"          
TOKENIZER_JSON = "python_tokenizer.json"   #
MAX_LEN = 512

In [37]:
def load_jsonl(path: str) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

@dataclass
class IfMaskMLMBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

class IfMaskMLMDataset(Dataset):
    """
    Turns {"input": func_with_<mask>, "target": condition_text} into MLM examples:
      - Tokenize input
      - Tokenize target (without special tokens)
      - Replace single <mask> token with N copies of mask-token (N=len(target_ids))
      - Labels = -100 everywhere except the mask span, where labels=target_ids
    """
    def __init__(self, rows: List[Dict], tok: PreTrainedTokenizerFast, max_len: int = 512):
        self.rows = rows
        self.tok = tok
        self.max_len = max_len
        self.mask_id = tok.convert_tokens_to_ids(tok.mask_token)

        cleaned: List[IfMaskMLMBatch] = []
        for r in rows:
            inp = r["input"]
            tgt = r["target"]

            # Tokenize input and target
            enc_inp = tok(inp, add_special_tokens=True, truncation=False)
            enc_tgt = tok(tgt, add_special_tokens=False)

            # Find the single <mask> position in input ids
            input_ids = enc_inp["input_ids"]
            try:
                mask_pos = input_ids.index(self.mask_id)
            except ValueError:
                continue

            # Build expanded sequence: input_ids with <mask> replaced by len(target_ids) masks
            tgt_ids = enc_tgt["input_ids"]
            if len(tgt_ids) == 0:
                continue

            expanded = (
                input_ids[:mask_pos]
                + [self.mask_id] * len(tgt_ids)
                + input_ids[mask_pos + 1 :]
            )

            # Truncate if too long - simple policy: skip long example to keep code short
            if len(expanded) > max_len:
                continue

            # Labels: -100 everywhere, fill target ids at mask span
            labels = [-100] * len(expanded)
            for i, tid in enumerate(tgt_ids):
                labels[mask_pos + i] = tid

            # Attention mask
            attn = [1] * len(expanded)

            cleaned.append(
                IfMaskMLMBatch(
                    input_ids=torch.tensor(expanded, dtype=torch.long),
                    attention_mask=torch.tensor(attn, dtype=torch.long),
                    labels=torch.tensor(labels, dtype=torch.long),
                )
            )

        self.examples = cleaned

    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        ex = self.examples[i]
        return {
            "input_ids": ex.input_ids,
            "attention_mask": ex.attention_mask,
            "labels": ex.labels,
        }
        
        
def collate_mlm(batch, pad_id: int, pad_mult: int | None = 8):
    """
    Pads input_ids with pad_id, attention_mask with 0, labels with -100.
    Optionally pad to multiple of pad_mult (e.g., 8) for speed on GPU.
    """
    input_ids      = [ex["input_ids"] for ex in batch]
    attention_mask = [ex["attention_mask"] for ex in batch]
    labels         = [ex["labels"] for ex in batch]

    input_ids      = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels         = pad_sequence(labels, batch_first=True, padding_value=-100)

    if pad_mult:
        L = input_ids.size(1)
        rem = (-L) % pad_mult
        if rem:
            pad_cols = (0, rem)
            input_ids      = torch.nn.functional.pad(input_ids, pad_cols, value=pad_id)
            attention_mask = torch.nn.functional.pad(attention_mask, pad_cols, value=0)
            labels         = torch.nn.functional.pad(labels, pad_cols, value=-100)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [4]:
def mlm_accuracy(eval_pred):
    logits, labels = eval_pred  # logits: (N, T, V), labels: (N, T)
    preds = logits.argmax(-1)
    mask = labels != -100
    total = mask.sum()
    if total == 0:
        return {"mlm_acc": 0.0}
    correct = (preds[mask] == labels[mask]).sum()
    return {"mlm_acc": (correct / total).item()}

In [5]:
# Tokenizer
tok = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER_JSON)
tok.add_special_tokens({
    "pad_token": "<pad>", "unk_token": "<unk>", "mask_token": "<mask>",
    "bos_token": "<s>", "eos_token": "</s>",
})

0

In [6]:
train_rows = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
val_rows   = load_jsonl(os.path.join(DATA_DIR, "validation.jsonl"))
test_rows  = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))

train_ds = IfMaskMLMDataset(train_rows, tok, MAX_LEN)
val_ds   = IfMaskMLMDataset(val_rows, tok, MAX_LEN)
test_ds  = IfMaskMLMDataset(test_rows, tok, MAX_LEN)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 271343 | Val: 30191 | Test: 33484


In [43]:
# print some examples
i = 1
ex = train_ds[i]
print("Example", i)
print("Labels:       ", ex["labels"])
print("Decoded Input:", tok.decode(ex["input_ids"].tolist()))
label_ids = ex["labels"].tolist()
decoded_labels = tok.decode([lid for lid in label_ids if lid != -100])
print("Decoded Target:", decoded_labels)
print()

Example 1
Labels:        tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100,  842,  489,   18, 1814,  291, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100])
Decoded Input: def Ġsearch ( self , Ġqueryset , Ġname , Ġvalue ): Ċ ĠĠĠĠĠĠĠ Ġif Ġ <mask> <mask> <mask> <mask> <mask> : Ċ ĠĠĠĠĠĠĠĠĠĠĠ Ġreturn Ġqueryset Ċ ĠĠĠĠĠĠĠ Ġreturn Ġqueryset . filter ( Ċ ĠĠĠĠĠĠĠĠĠĠĠ Ġq ( circuit __ cid __ icontains = value ) Ġ| Ċ ĠĠĠĠĠĠĠĠĠĠĠ Ġq ( x connect _ id __ icontains = value ) Ġ| Ċ ĠĠĠĠĠĠĠĠĠĠĠ Ġq ( pp _ info __ icontains = value ) Ġ| Ċ ĠĠĠĠĠĠĠĠĠĠĠ Ġq ( desc

In [9]:
model = BertForMaskedLM.from_pretrained(MODEL_DIR)

data_collator = lambda batch: collate_mlm(batch, pad_id=tok.pad_token_id, pad_mult=8)

args = TrainingArguments(
    output_dir="if_mlm_finetuned",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    num_train_epochs=3,
    logging_steps=10_000,
    do_eval=True,
    save_steps=10_000,
    fp16=torch.cuda.is_available(),
    report_to="none",   # remove if your transformers is too old
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=data_collator,
    compute_metrics=mlm_accuracy,
)

trainer.train()

trainer.save_model("if_mlm_finetuned")
tok.save_pretrained("if_mlm_finetuned")
print("[DONE] Saved fine-tuned model to if_mlm_finetuned")

/tmp/ipykernel_3192164/1266346572.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10000,3.118000
20000,2.396300


[DONE] Saved fine-tuned model to if_mlm_finetuned


In [10]:
# Evaluation
with torch.no_grad():
    list_test_acc = []
    for test_batch in test_ds:
        test_batch = {k: v.unsqueeze(0).to(trainer.args.device) for k, v in test_batch.items()}
        outputs = model(**test_batch)
        
        logits = outputs.logits
        preds = logits.argmax(-1)
    
        acc = mlm_accuracy((logits.cpu().numpy(), test_batch['labels'].cpu().numpy()))
        list_test_acc.append(acc['mlm_acc'])
        
    mean_test_acc = np.mean(list_test_acc)
    print(f"Test MLM Accuracy: {mean_test_acc:.4f}")

Test MLM Accuracy: 0.6160


In [41]:
def predict_if_condition(func_with_mask: str, tokenizer: PreTrainedTokenizerFast, model: BertForMaskedLM, top_k: int = 5):
    """
    Given a function text with a single <mask>, predict the masked condition.
    Returns the top_k predicted condition strings with their scores.
    """ 
    model.eval()
    inputs = tokenizer(func_with_mask, return_tensors="pt")
    print("Input IDs:", inputs["input_ids"].tolist())
    
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    print("Mask token index:", mask_token_index.item())
    
    # put inputs to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        mask_token_logits = logits[0, mask_token_index, :]
        top_k_ids = torch.topk(mask_token_logits, top_k, dim=1).indices[0].tolist()
        
    predicted_conditions = []
    for token_id in top_k_ids:
        token = tokenizer.decode([token_id])
        score = F.softmax(mask_token_logits, dim=1)[0, token_id].item()
        predicted_conditions.append((token, score))
        
    return predicted_conditions

In [47]:
example = """\
def check_x(x):
    if <mask>:
        return "positive"
    else:
        return "negative"
"""

predictions = predict_if_condition(example, tok, model, top_k=1)

Input IDs: [[1285, 730, 41, 66, 12, 66, 261, 154, 811, 283, 174, 4, 30, 154, 2018, 296, 234, 5942, 6, 154, 811, 460, 30, 154, 2018, 296, 234, 2886, 6, 154]]
Mask token index: 11


In [48]:
predictions

[('x', 0.9808374047279358)]

In [ ]:

print("Predicted condition:", tok.decode(predictions[0], skip_special_tokens=True))

Input IDs: [[1285, 730, 41, 66, 12, 66, 261, 154, 811, 283, 174, 4, 30, 154, 2018, 296, 234, 5942, 6, 154, 811, 460, 30, 154, 2018, 296, 234, 2886, 6, 154]]
Mask token index: 11


TypeError: argument 'ids': 'str' object cannot be interpreted as an integer